# FIT5217 A2 Task 2 Colab Training Notebook

This notebook trains Task 2.1 T5 LoRA with three configurations and Task 2.2 GPT-2 LoRA with one configuration on a Colab **L4 GPU**.

Drive root reused from Task 1:

`/content/drive/MyDrive/fit5217_a2/`

Expected outputs:
- Checkpoints: `/content/drive/MyDrive/fit5217_a2/checkpoints/`
- Predictions: `/content/drive/MyDrive/fit5217_a2/outputs/predictions/`
- Metrics: `/content/drive/MyDrive/fit5217_a2/outputs/metrics/`

Training-time estimate on L4 with the current budget configs:
- T5 LoRA config1-3, 3 epochs each: roughly 3-6 hours total
- GPT-2 LoRA, 3 epochs: roughly 1.5-3 hours
- Full test inference + BERTScore: roughly 0.5-1.5 hours

Planning estimate: **5-10 hours** total, about **25-55 Colab units** depending on runtime speed and reruns.


In [ ]:
# GPU check
!nvidia-smi

import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / (1024 ** 3):.1f} GB")
else:
    print("No CUDA GPU detected. Use Runtime > Change runtime type > GPU, preferably L4.")


In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Clone or update repository
from pathlib import Path

REPO_URL = 'https://github.com/jche0572/fit5217-a2.git'
REPO_DIR = Path('/content/fit5217-a2')

%cd /content
if REPO_DIR.exists():
    %cd /content/fit5217-a2
    !git pull
else:
    !git clone {REPO_URL}
    %cd /content/fit5217-a2

!git rev-parse --short HEAD


In [ ]:
# Install dependencies and verify imports
%cd /content/fit5217-a2
!pip install -q -r requirements-colab.txt

import peft
import transformers
import sentencepiece
import torch

print('peft:', peft.__version__)
print('transformers:', transformers.__version__)
print('sentencepiece: OK')
print('torch:', torch.__version__)


In [ ]:
# Data and Drive-path preparation
from pathlib import Path
import json
import pickle
import shutil
import pandas as pd

REPO_DIR = Path('/content/fit5217-a2')
DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
LOCAL_OUTPUTS = REPO_DIR / 'outputs'
LOCAL_CHECKPOINTS = REPO_DIR / 'checkpoints'

DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUTS / 'predictions').mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUTS / 'metrics').mkdir(parents=True, exist_ok=True)

# Copy Task 1 processed artifacts from Drive to local ./outputs as requested.
(LOCAL_OUTPUTS / 'processed').mkdir(parents=True, exist_ok=True)
for src in [DRIVE_OUTPUTS / 'vocab.pkl'] + sorted((DRIVE_OUTPUTS / 'processed').glob('*.pkl')):
    dst = LOCAL_OUTPUTS / src.relative_to(DRIVE_OUTPUTS)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied {src} -> {dst}')

# Make checkpoints Drive-backed while preserving the scripts' relative output_dir fields.
if LOCAL_CHECKPOINTS.exists() and not LOCAL_CHECKPOINTS.is_symlink():
    backup = REPO_DIR / 'checkpoints_repo_backup'
    if not backup.exists():
        shutil.move(str(LOCAL_CHECKPOINTS), str(backup))
    else:
        shutil.rmtree(LOCAL_CHECKPOINTS)
if LOCAL_CHECKPOINTS.is_symlink() or LOCAL_CHECKPOINTS.exists():
    if LOCAL_CHECKPOINTS.resolve() != DRIVE_CHECKPOINTS.resolve():
        LOCAL_CHECKPOINTS.unlink()
if not LOCAL_CHECKPOINTS.exists():
    LOCAL_CHECKPOINTS.symlink_to(DRIVE_CHECKPOINTS, target_is_directory=True)
print('checkpoints ->', LOCAL_CHECKPOINTS.resolve())

# Raw CSVs are needed for T2 fine-tuning. Prefer Drive copy if present; otherwise use repo copy.
local_raw = REPO_DIR / 'data' / 'raw'
drive_raw = DRIVE_ROOT / 'data' / 'raw'
local_raw.mkdir(parents=True, exist_ok=True)
if drive_raw.exists():
    for src in sorted(drive_raw.glob('*.csv')):
        shutil.copy2(src, local_raw / src.name)
        print(f'copied raw {src} -> {local_raw / src.name}')

for split in ['train', 'dev', 'test']:
    path = local_raw / f'{split}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Upload raw CSVs to {drive_raw} or include them in the repo runtime.')
    df = pd.read_csv(path)
    print(f'{split}: {len(df):,} rows')

with open(LOCAL_OUTPUTS / 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
print('vocab size:', len(vocab['word2idx']))
for split in ['train', 'dev', 'test']:
    with open(LOCAL_OUTPUTS / 'processed' / f'{split}_ids.pkl', 'rb') as f:
        rows = pickle.load(f)
    print(f'processed {split}: {len(rows):,} examples')


## T2.1 T5 LoRA Training

The three YAML configs use the same base model `google-t5/t5-small` and differ only in LoRA settings. Checkpoints are saved through the Drive-backed `checkpoints/` symlink, so rerunning a cell resumes from `last/`.


In [ ]:
# T5 config1 training
import subprocess
import time
from pathlib import Path

%cd /content/fit5217-a2
start = time.time()
cmd = ['python', 'scripts/train_t2_t5.py', '--config', 'configs/t2_t5_config1.yaml']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
elapsed = time.time() - start
print(f'T5 config1 training elapsed: {elapsed/3600:.2f} hours')
print(f'Estimated L4 units for this cell: {elapsed/3600*4:.1f}-{elapsed/3600*6:.1f}')
print('Checkpoint directory:', Path('/content/drive/MyDrive/fit5217_a2/checkpoints') / Path('configs/t2_t5_config1.yaml').stem)


In [ ]:
# T5 config2 training
import subprocess
import time
from pathlib import Path

%cd /content/fit5217-a2
start = time.time()
cmd = ['python', 'scripts/train_t2_t5.py', '--config', 'configs/t2_t5_config2.yaml']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
elapsed = time.time() - start
print(f'T5 config2 training elapsed: {elapsed/3600:.2f} hours')
print(f'Estimated L4 units for this cell: {elapsed/3600*4:.1f}-{elapsed/3600*6:.1f}')
print('Checkpoint directory:', Path('/content/drive/MyDrive/fit5217_a2/checkpoints') / Path('configs/t2_t5_config2.yaml').stem)


In [ ]:
# T5 config3 training
import subprocess
import time
from pathlib import Path

%cd /content/fit5217-a2
start = time.time()
cmd = ['python', 'scripts/train_t2_t5.py', '--config', 'configs/t2_t5_config3.yaml']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
elapsed = time.time() - start
print(f'T5 config3 training elapsed: {elapsed/3600:.2f} hours')
print(f'Estimated L4 units for this cell: {elapsed/3600*4:.1f}-{elapsed/3600*6:.1f}')
print('Checkpoint directory:', Path('/content/drive/MyDrive/fit5217_a2/checkpoints') / Path('configs/t2_t5_config3.yaml').stem)


In [ ]:
# T5 test inference + metrics for all three configs
import json
import re
import sys
import time
from pathlib import Path

import contextlib
import io
import nltk
import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.models.t5_lora import format_t5_input

DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
PRED_DIR = DRIVE_ROOT / 'outputs' / 'predictions'
METRIC_DIR = DRIVE_ROOT / 'outputs' / 'metrics'
PRED_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def recipe_to_text(recipe_list):
    return ' '.join(str(x).strip() for x in recipe_list if str(x).strip())

def run_teacher_metrics(gold_recipes, pred_recipes):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        evaluate2(gold_recipes, pred_recipes)
    text = buffer.getvalue()
    print(text)
    metrics = {}
    patterns = {
        'BLEU-4': r'BLEU-4:\s*([0-9.]+)',
        'METEOR': r'METEOR:\s*([0-9.]+)',
        'BERTScore': r'BERTScore:\s*([0-9.]+)',
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        metrics[key] = float(match.group(1)) if match else None
    return metrics

def patch_eval_metrics():
    eval_path = REPO_DIR / 'eval_metrics.py'
    text = eval_path.read_text()
    patched = re.sub(r'nltk_cache_dir\s*=\s*\[YOUR_DIR\]', "nltk_cache_dir = '/content/nltk_data'", text)
    if patched != text:
        eval_path.write_text(patched)

test_df = pd.read_csv(REPO_DIR / 'data' / 'raw' / 'test.csv')
test_ingredients = [json.loads(x) for x in test_df['Ingredients']]
gold_recipes = [recipe_to_text(json.loads(x)) for x in test_df['Recipe']]

configs = ['t2_t5_config1', 't2_t5_config2', 't2_t5_config3']
all_metrics = {}
patch_eval_metrics()
from eval_metrics import evaluate2

for config_name in configs:
    print(f'\n=== Inference: {config_name} ===')
    ckpt_dir = DRIVE_ROOT / 'checkpoints' / config_name / 'best'
    if not ckpt_dir.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt_dir}')
    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
    base = AutoModelForSeq2SeqLM.from_pretrained('google-t5/t5-small')
    model = PeftModel.from_pretrained(base, ckpt_dir).to(DEVICE)
    model.eval()

    preds = []
    batch_size = 32
    start = time.time()
    with torch.no_grad():
        for start_idx in range(0, len(test_ingredients), batch_size):
            batch_ingredients = test_ingredients[start_idx:start_idx + batch_size]
            prompts = [format_t5_input(x) for x in batch_ingredients]
            inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
            output_ids = model.generate(**inputs, max_new_tokens=128, num_beams=1, do_sample=False)
            preds.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
            if (start_idx // batch_size) % 10 == 0:
                print(f'{config_name}: {min(start_idx + batch_size, len(test_ingredients))}/{len(test_ingredients)}')
    print(f'{config_name} inference elapsed: {(time.time()-start)/60:.1f} min')

    pred_path = PRED_DIR / f't2_t5_{config_name}_test.json'
    with open(pred_path, 'w') as f:
        json.dump([{'gold': g, 'prediction': p} for g, p in zip(gold_recipes, preds)], f, indent=2)
    print('Saved predictions:', pred_path)

    print('Teacher eval_metrics.evaluate2 output:')
    metrics = run_teacher_metrics(gold_recipes, preds)
    all_metrics[config_name] = metrics
    print(metrics)

metrics_path = METRIC_DIR / 't2_t5_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('Saved metrics:', metrics_path)
pd.DataFrame.from_dict(all_metrics, orient='index')


## T2.2 GPT-2 LoRA Training

The GPT-2 decoder-only setup trains on a single concatenated sequence. Ingredient prompt tokens are masked with `-100` in labels, so loss is computed only on the generated recipe continuation.


In [ ]:
# GPT-2 LoRA training
import subprocess
import time
from pathlib import Path

%cd /content/fit5217-a2
start = time.time()
cmd = ['python', 'scripts/train_t2_gpt2.py', '--config', 'configs/t2_gpt2.yaml']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
elapsed = time.time() - start
print(f'GPT-2 training elapsed: {elapsed/3600:.2f} hours')
print(f'Estimated L4 units for this cell: {elapsed/3600*4:.1f}-{elapsed/3600*6:.1f}')
print('Checkpoint directory:', Path('/content/drive/MyDrive/fit5217_a2/checkpoints/t2_gpt2'))


In [ ]:
# GPT-2 test inference with greedy + beam decoding, then metrics
import json
import re
import sys
import time
from pathlib import Path

import contextlib
import io
import nltk
import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.models.gpt2_lora import generate_recipes

DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
PRED_DIR = DRIVE_ROOT / 'outputs' / 'predictions'
METRIC_DIR = DRIVE_ROOT / 'outputs' / 'metrics'
PRED_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def recipe_to_text(recipe_list):
    return ' '.join(str(x).strip() for x in recipe_list if str(x).strip())

def run_teacher_metrics(gold_recipes, pred_recipes):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        evaluate2(gold_recipes, pred_recipes)
    text = buffer.getvalue()
    print(text)
    metrics = {}
    patterns = {
        'BLEU-4': r'BLEU-4:\s*([0-9.]+)',
        'METEOR': r'METEOR:\s*([0-9.]+)',
        'BERTScore': r'BERTScore:\s*([0-9.]+)',
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        metrics[key] = float(match.group(1)) if match else None
    return metrics

def patch_eval_metrics():
    eval_path = REPO_DIR / 'eval_metrics.py'
    text = eval_path.read_text()
    patched = re.sub(r'nltk_cache_dir\s*=\s*\[YOUR_DIR\]', "nltk_cache_dir = '/content/nltk_data'", text)
    if patched != text:
        eval_path.write_text(patched)

test_df = pd.read_csv(REPO_DIR / 'data' / 'raw' / 'test.csv')
test_ingredients = [json.loads(x) for x in test_df['Ingredients']]
gold_recipes = [recipe_to_text(json.loads(x)) for x in test_df['Recipe']]

ckpt_dir = DRIVE_ROOT / 'checkpoints' / 't2_gpt2' / 'best'
if not ckpt_dir.exists():
    raise FileNotFoundError(f'Missing checkpoint: {ckpt_dir}')
tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
base = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')
base.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base, ckpt_dir).to(DEVICE)
model.eval()

patch_eval_metrics()
from eval_metrics import evaluate2

strategy_kwargs = {
    'greedy': {'max_new_tokens': 128},
    'beam': {'max_new_tokens': 128, 'num_beams': 4},
}
all_metrics = {}
for strategy, kwargs in strategy_kwargs.items():
    print(f'\n=== GPT-2 inference: {strategy} ===')
    preds = []
    batch_size = 16 if strategy == 'beam' else 32
    start = time.time()
    with torch.no_grad():
        for start_idx in range(0, len(test_ingredients), batch_size):
            batch_ingredients = test_ingredients[start_idx:start_idx + batch_size]
            preds.extend(generate_recipes(model, tokenizer, batch_ingredients, decoding_strategy=strategy, **kwargs))
            if (start_idx // batch_size) % 10 == 0:
                print(f'{strategy}: {min(start_idx + batch_size, len(test_ingredients))}/{len(test_ingredients)}')
    print(f'{strategy} inference elapsed: {(time.time()-start)/60:.1f} min')

    pred_path = PRED_DIR / f't2_gpt2_{strategy}_test.json'
    with open(pred_path, 'w') as f:
        json.dump([{'gold': g, 'prediction': p} for g, p in zip(gold_recipes, preds)], f, indent=2)
    print('Saved predictions:', pred_path)

    print('Teacher eval_metrics.evaluate2 output:')
    metrics = run_teacher_metrics(gold_recipes, preds)
    all_metrics[strategy] = metrics
    print(metrics)

metrics_path = METRIC_DIR / 't2_gpt2_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('Saved metrics:', metrics_path)
pd.DataFrame.from_dict(all_metrics, orient='index')


In [ ]:
# T2 full comparison summary
import json
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
METRIC_DIR = DRIVE_ROOT / 'outputs' / 'metrics'
rows = []

t5_path = METRIC_DIR / 't2_t5_metrics.json'
if t5_path.exists():
    t5_metrics = json.load(open(t5_path))
    for name, metrics in t5_metrics.items():
        rows.append({'model': f'T5 LoRA {name}', 'decoding': 'greedy', **metrics})
else:
    print('Missing', t5_path)

gpt2_path = METRIC_DIR / 't2_gpt2_metrics.json'
if gpt2_path.exists():
    gpt2_metrics = json.load(open(gpt2_path))
    for strategy, metrics in gpt2_metrics.items():
        rows.append({'model': 'GPT-2 LoRA', 'decoding': strategy, **metrics})
else:
    print('Missing', gpt2_path)

summary = pd.DataFrame(rows)
display(summary)
summary_path = METRIC_DIR / 't2_all_metrics.csv'
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
